# Manipulação e Documentação do Dado
### Pandas, Dicionário e Catálogo de Dados

**Notebook de prática**

> Essa prática usa uma base fictícia — mas realista e "suja" — de
> **agendamentos de consultas em uma rede de clínicas** (faltas, turno, faixa etária, CEP do paciente x
> CEP da unidade).

**Roteiro:**

1. Do dado à pergunta de negócio
2. Pandas — leitura e tipos
3. Seleção e filtros
4. `groupby`
5. `merge`
6. Primeiro diagnóstico de nulos
7. Dicionário de dados
8. Catálogo, linhagem e governança

---


In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

print("pandas:", pd.__version__)
print("numpy :", np.__version__)

pandas: 3.0.4
numpy : 2.5.0


## A base usada neste notebook

A prática usa uma base de **agendamentos de uma rede de clínicas**, com duas tabelas fornecidas em
arquivos CSV separados (fora deste notebook):

- `agendamentos.csv` — uma linha por agendamento (grão = 1 agendamento), salva com os defeitos típicos de
  uma exportação brasileira: separador `;`, encoding `latin1`, decimal com vírgula, nulos disfarçados
  (`"-"`, `"não informado"`, `"999"`...), variações de escrita (`"SP"`, `"sp"`, `" SP"`) e uma
  concentração de nulos em janeiro/2024 (simulando troca de sistema).
- `unidades.csv` — uma linha por unidade de atendimento (tabela de apoio para os exemplos de `merge`).

**Antes de rodar a célula abaixo**, salve os dois arquivos na mesma pasta deste notebook (ou anote o
caminho completo até eles).


In [ ]:
from pathlib import Path

caminho_agendamentos = input(
    "Informe o caminho para o arquivo agendamentos.csv (ENTER para usar o padrão './agendamentos.csv'): "
).strip()
if caminho_agendamentos == "":
    caminho_agendamentos = "agendamentos.csv"

caminho_agendamentos = Path(caminho_agendamentos)
if not caminho_agendamentos.exists():
    raise FileNotFoundError(
        f"Não encontrei o arquivo em '{caminho_agendamentos}'. "
        "Confira se ele está na mesma pasta deste notebook ou informe o caminho completo."
    )

print("Lendo arquivo de:", caminho_agendamentos.resolve())

Lendo arquivo de: C:\Users\Clara Salomão\Downloads\files\agendamentos.csv


---
# 1. Do dado à pergunta de negócio

**Contexto:** antes desta etapa, o grupo já deve ter escolhido uma base do projeto e produzido uma
especificação inicial dela (contrato de interface, dicionário de campos, pipeline). Agora essa base sai
do papel: vamos carregá-la,
olhá-la de verdade e responder a uma pergunta incômoda — **esta base responde à pergunta que o seu PI se
propôs a responder?**

### O que é uma pergunta de negócio

Uma pergunta de negócio não é um tema. É uma frase que, respondida, **muda uma decisão**. Ela tem quatro partes:

| Parte | O que é |
|---|---|
| **A métrica** | O que exatamente está sendo medido — e em que unidade |
| **O recorte** | De quem, em que período, em que lugar |
| **A comparação** | Contra o quê? Um período anterior, outro grupo, uma meta. Número sozinho não informa |
| **A decisão que ela habilita** | Se a resposta for A, faz-se isso; se for B, faz-se aquilo. Se nenhuma resposta muda nada, a pergunta é decorativa |

### Vaga ou respondível

| Pergunta vaga | O que falta | Pergunta respondível |
|---|---|---|
| Como está a satisfação dos clientes? | Métrica, recorte, comparação | A nota média de atendimento no 1º semestre caiu em relação ao mesmo período do ano anterior, por unidade? |
| Queremos entender as faltas em consultas. | É um tema, não uma pergunta | Qual a taxa de falta por faixa etária e por turno, e em quais combinações ela supera 20%? |
| Analisar vendas por região. | Nenhuma decisão associada | Quais regiões ficaram abaixo da meta trimestral, e qual a diferença em reais para atingi-la? |
| Usar IA para melhorar processos. | Tudo | Quantos chamados de nível 1 poderiam ser resolvidos sem atendente, considerando os que hoje se resolvem com uma única resposta padrão? |

### O caminho reverso

A tentação é começar pela base e ver o que dá para fazer com ela. **O caminho correto é o inverso:**
partir da pergunta → métrica (como se calcula) → colunas (quais campos) → base (ela tem esses campos, com
a granularidade certa?).

---
### Exercício 1

Para o **nosso** projeto fictício (rede de clínicas), a pergunta de negócio é:

> *"Qual a taxa de falta por turno e por faixa etária, e em quais combinações ela supera 20%?"*

Monte a **matriz pergunta → métrica → colunas necessárias → operação em Pandas → tem na base?**, exatamente
como no exemplo visto na teoria. Inclua também uma segunda pergunta auxiliar sobre o efeito da distância
paciente-unidade (que exige uma base externa de CEP), para mostrar um caso em que a base **não** responde
sozinha.

### Resolução


,pergunta,metrica,colunas_necessarias,operacao_pandas,tem_na_base
0,Taxa de falta por turno e faixa etária,faltas / agendamentos,"situacao, turno, idade","groupby(['turno','faixa_etaria']) + agg",Sim
1,Efeito da distância paciente-unidade na falta,taxa de falta por faixa de distância (km),"cep_paciente, cep_unidade",merge externo (base de CEP/IBGE) + cálculo de ...,Não — exige base externa de CEP


---
# 2. Pandas — leitura e tipos

A maior parte dos problemas de uma análise nasce na primeira linha de código. Parâmetros de `read_csv` que importam:

| Parâmetro | Para que serve | Quando você precisa |
|---|---|---|
| `sep` | Separador de campos | Sempre que não for vírgula — no Brasil, quase sempre |
| `encoding` | Como os bytes viram caracteres | Sempre. Nunca confie no padrão |
| `dtype` | Força o tipo de cada coluna | Para impedir que o Pandas adivinhe errado |
| `decimal` / `thousands` | Vírgula decimal e ponto de milhar | Todo arquivo gerado em padrão brasileiro |
| `parse_dates` | Converte colunas em datetime na leitura | Quando o formato da data é reconhecível |
| `na_values` | Quais textos devem virar NaN | Bases com "-", "N/A", "não informado" |
| `usecols` | Lê só as colunas listadas | Arquivos largos: economiza memória e tempo |
| `nrows` | Lê só as primeiras n linhas | Inspeção rápida antes de carregar tudo |
| `skiprows` | Pula linhas do início | Arquivos com cabeçalho decorativo |

### A leitura em duas etapas

Recomendação prática: **leia tudo como texto na primeira vez**. Deixe a conversão de tipos ser uma
decisão consciente, e não um palpite.


In [ ]:
# 1) espiar antes de carregar
amostra = pd.read_csv(caminho_agendamentos, sep=";", encoding="latin1", nrows=50)
amostra.head()

,id_agendamento,id_paciente,data_nascimento,idade,sexo,uf,cep_paciente,id_unidade,turno,dt_agendamento,dt_consulta,situacao,valor_consulta,renda_familiar
0,AG100000,P00277,20/05/1981,43,M,RJ,68642-706,U01,Tarde,15/05/2024 15:15,22/05/2024 15:15,Cancelada,"147,15",2439.09
1,AG100001,P00337,17/12/1983,40,M,SP,36504-994,U07,Manha,14/01/2024 12:45,23/01/2024 12:45,Realizada,"207,00",2310.42
2,AG100002,P00750,28/03/1965,59,M,MG,74909-180,U06,Manha,24/05/2024 13:45,28/05/2024 13:45,Realizada,"264,04",2925.54
3,AG100003,P00743,08/01/2015,9,F,RJ,92316-738,U04,Manha,03/05/2024 17:45,27/05/2024 17:45,Falta,"140,07",2371.99
4,AG100004,P00786,14/04/1966,58,F,SP,31777-947,U04,Tarde,14/03/2024 18:30,02/04/2024 18:30,Realizada,"275,33",5487.68


In [ ]:
# 2) carregar tudo como texto, controlando os nulos
df = pd.read_csv(
    caminho_agendamentos,
    sep=";",
    encoding="latin1",
    dtype=str,                       # nada de adivinhacao
    na_values=["", "-", "N/A", "NA", "nao informado", "999", " "], #tratamento/controle dos nulos
    keep_default_na=True,
)

print(df.shape)
df.info()

(1500, 14)
<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id_agendamento   1500 non-null   str  
 1   id_paciente      1500 non-null   str  
 2   data_nascimento  1442 non-null   str  
 3   idade            1456 non-null   str  
 4   sexo             1500 non-null   str  
 5   uf               1500 non-null   str  
 6   cep_paciente     1440 non-null   str  
 7   id_unidade       1500 non-null   str  
 8   turno            1500 non-null   str  
 9   dt_agendamento   1500 non-null   str  
 10  dt_consulta      1500 non-null   str  
 11  situacao         1500 non-null   str  
 12  valor_consulta   1500 non-null   str  
 13  renda_familiar   1451 non-null   str  
dtypes: str(14)
memory usage: 308.6 KB


### O primeiro contato com a base

Use esses comandos, nesta ordem, antes de qualquer análise. `head()` mostra o começo do arquivo
(geralmente a parte mais limpa); `sample()` mostra o meio, onde costuma estar o problema.


In [ ]:
print("shape:", df.shape)
df.head(10)

shape: (1500, 14)


,id_agendamento,id_paciente,data_nascimento,idade,sexo,uf,cep_paciente,id_unidade,turno,dt_agendamento,dt_consulta,situacao,valor_consulta,renda_familiar
0,AG100000,P00277,20/05/1981,43,M,RJ,68642-706,U01,Tarde,15/05/2024 15:15,22/05/2024 15:15,Cancelada,"147,15",2439.09
1,AG100001,P00337,17/12/1983,40,M,SP,36504-994,U07,Manha,14/01/2024 12:45,23/01/2024 12:45,Realizada,"207,00",2310.42
2,AG100002,P00750,28/03/1965,59,M,MG,74909-180,U06,Manha,24/05/2024 13:45,28/05/2024 13:45,Realizada,"264,04",2925.54
3,AG100003,P00743,08/01/2015,9,F,RJ,92316-738,U04,Manha,03/05/2024 17:45,27/05/2024 17:45,Falta,"140,07",2371.99
4,AG100004,P00786,14/04/1966,58,F,SP,31777-947,U04,Tarde,14/03/2024 18:30,02/04/2024 18:30,Realizada,"275,33",5487.68
5,AG100005,P00417,08/06/1986,38,M,SP,25587-643,U07,Noite,31/03/2024 13:30,27/04/2024 13:30,Falta,"197,36",99999999
6,AG100006,P00294,04/05/1954,70,M,São Paulo,67341-962,U01,Noite,14/01/2024 09:45,15/01/2024 09:45,Realizada,"160,50",3221.9
7,AG100007,P00449,05/02/1931,93,F,São Paulo,57439-993,U06,Tarde,07/02/2024 15:45,26/02/2024 15:45,Realizada,"139,80",1800.91
8,AG100008,P00712,NaN,21,M,SP,29147-800,U02,Tarde,14/02/2024 15:15,02/03/2024 15:15,Realizada,"169,95",4308.48
9,AG100009,P00559,13/12/1952,71,M,MG,25292-998,U01,Manha,07/06/2024 10:45,26/06/2024 10:45,Realizada,"92,01",2282.59


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id_agendamento   1500 non-null   str  
 1   id_paciente      1500 non-null   str  
 2   data_nascimento  1442 non-null   str  
 3   idade            1456 non-null   str  
 4   sexo             1500 non-null   str  
 5   uf               1500 non-null   str  
 6   cep_paciente     1440 non-null   str  
 7   id_unidade       1500 non-null   str  
 8   turno            1500 non-null   str  
 9   dt_agendamento   1500 non-null   str  
 10  dt_consulta      1500 non-null   str  
 11  situacao         1500 non-null   str  
 12  valor_consulta   1500 non-null   str  
 13  renda_familiar   1451 non-null   str  
dtypes: str(14)
memory usage: 308.6 KB


In [ ]:
# describe() so enxerga numerico de verdade -- aqui tudo ainda e string (dtype=str),
# entao describe() por enquanto so mostra estatisticas de "objeto" (contagem, unicos, top, freq)
df.describe()

       id_agendamento id_paciente data_nascimento idade  sexo    uf cep_paciente id_unidade  turno    dt_agendamento       dt_consulta  \
count            1500        1500            1442  1456  1500  1500         1440       1500   1500              1500              1500   
unique           1500         716            1392    95     3    11         1440          8     10              1378              1371   
top          AG100000      P00803      01/01/1900    57     M    SP    68642-706        U07  Manha  24/01/2024 17:00  02/04/2024 18:30   
freq                1           7              17    23   730   784            1        200    568                 4                 3   

         situacao valor_consulta renda_familiar  
count        1500           1500           1451  
unique          3           1422           1365  
top     Realizada          60,00          300.0  
freq         1090              6             32  

In [ ]:
df.sample(5, random_state=7)

    id_agendamento id_paciente data_nascimento idade sexo  uf cep_paciente id_unidade   turno    dt_agendamento       dt_consulta  \
547       AG100547      P00042      20/02/1961    63    M  RJ    56292-677        U05   Noite  29/03/2024 17:30  19/04/2024 17:30   
984       AG100984      P00204      18/10/1976    47    F  SP    17831-191        U05   Tarde  10/01/2024 09:30  21/01/2024 09:30   
712       AG100712      P00163      12/10/1980    43    M  SP    99587-103        U06  MANHA   01/03/2024 16:30  08/03/2024 16:30   
638       AG100638      P00519      11/03/1985    39    F  SP    04996-407        U07   Tarde  23/04/2024 09:30  10/05/2024 09:30   
471       AG100471      P00228      27/03/1971    53    M  SP    47164-375        U07   Manha  21/06/2024 11:00  20/07/2024 11:00   

      situacao valor_consulta renda_familiar  
547  Realizada         185,12         3776.2  
984  Realizada         162,26         883.98  
712  Realizada         170,25        3231.04  
638  Realizad

### Os tipos do Pandas

| Tipo | O que guarda | Cuidado |
|---|---|---|
| `int64` / `float64` | Números | `int64` não aceita nulo: um NaN converte a coluna inteira em float |
| `object` | Texto, ou qualquer mistura | É o tipo "não sei o que é isto". Toda coluna nova chega assim |
| `category` | Texto com poucos valores distintos | Economiza muita memória; exige `observed=` no groupby |
| `datetime64` | Data e hora | Libera `.dt.year`, `.dt.weekday` e aritmética de datas |
| `bool` | Verdadeiro ou falso | Não aceita nulo; use `boolean` se houver |
| `Int64` (maiúsculo) | Inteiro que aceita nulo | Preserva o inteiro sem virar `float` por causa de um NaN |

### Convertendo com intenção

Cada conversão é uma decisão sobre o que fazer com o que não converte. O parâmetro `errors` define isso:
`errors="coerce"` transforma o inconversível em `NaN`; `errors="raise"` interrompe; **nunca use
`errors="ignore"`**, ele devolve a coluna original em silêncio.

---
### Exercício 2

Converta as colunas de `df` com intenção:
- `valor_consulta`: texto em formato brasileiro (`"1.234,56"`) → número
- `data_nascimento` e `dt_agendamento`/`dt_consulta`: texto → `datetime64`, com formato explícito
- `idade`: texto → `Int64` (aceita nulo)
- `renda_familiar`: texto → número, tratando a sentinela numérica `99999999`/`-1` como ausência
- `uf`, `turno`, `situacao`, `sexo`: texto → `category`

Ao final, confira **o que se perdeu** em cada conversão.

### Resolução


In [ ]:
df_tipado = df.copy()

# valor: texto brasileiro -> numero


# datas: formato explicito evita interpretacao errada de dia e mes


# idade: inteiro que aceita nulo


# renda: sentinelas numericas viram nulo antes da conversao


# categorias: memoria e semantica


# SEMPRE conferir o que se perdeu na conversao


valor_consulta - nao converteram: 0
data_nascimento - nao converteram: 58
idade - nulos (originais + sentinelas + disfarces): 44
renda_familiar - nulos (originais + sentinelas): 105
<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id_agendamento   1500 non-null   str           
 1   id_paciente      1500 non-null   str           
 2   data_nascimento  1442 non-null   datetime64[us]
 3   idade            1456 non-null   Int64         
 4   sexo             1500 non-null   category      
 5   uf               1500 non-null   category      
 6   cep_paciente     1440 non-null   str           
 7   id_unidade       1500 non-null   str           
 8   turno            1500 non-null   category      
 9   dt_agendamento   1500 non-null   datetime64[us]
 10  dt_consulta      1500 non-null   datetime64[us]
 11  situacao         1500 non-nu

---
# 3. Seleção e filtros

### `loc` e `iloc`

Duas formas de acessar. A confusão entre elas é um erro comum em Pandas.

| Operação | Comando | Observação |
|---|---|---|
| Uma coluna (Series) | `df["nome"]` | Devolve Series, não DataFrame |
| Várias colunas | `df[["nome","uf"]]` | Dois colchetes devolvem DataFrame |
| Linha pelo rótulo do índice | `df.loc[10]` | O rótulo, não a posição |
| Linha pela posição | `df.iloc[10]` | A 11ª linha, seja qual for o índice |
| Intervalo com `loc` | `df.loc[5:10]` | Inclui o 10 — diferente do Python puro |
| Intervalo com `iloc` | `df.iloc[5:10]` | Exclui o 10 — como o Python puro |
| Célula específica | `df.loc[5,"nome"]` | Cruzamento de linha e coluna |
| Filtro + colunas | `df.loc[df["uf"]=="SP", ["nome","valor"]]` | O padrão que você mais vai usar |


In [ ]:
print(type(df_tipado["id_paciente"]).__name__, "->", "df_tipado['id_paciente']")
print(type(df_tipado[["id_paciente","uf"]]).__name__, "->", "df_tipado[['id_paciente','uf']]")
print()
print("df_tipado.loc[5]:\n", df_tipado.loc[5][["id_agendamento","turno","situacao"]])
print()
print("df_tipado.iloc[5] (mesma coisa aqui pois o indice e 0..N-1 por padrao):\n",
      df_tipado.iloc[5][["id_agendamento","turno","situacao"]])

Series -> df_tipado['id_paciente']
DataFrame -> df_tipado[['id_paciente','uf']]

df_tipado.loc[5]:
 id_agendamento    AG100005
turno                Noite
situacao             Falta
Name: 5, dtype: object

df_tipado.iloc[5] (mesma coisa aqui pois o indice e 0..N-1 por padrao):
 id_agendamento    AG100005
turno                Noite
situacao             Falta
Name: 5, dtype: object


In [ ]:
print("loc[5:10] inclui o 10 -> linhas:", len(df_tipado.loc[5:10]))
print("iloc[5:10] exclui o 10 -> linhas:", len(df_tipado.iloc[5:10]))

loc[5:10] inclui o 10 -> linhas: 6
iloc[5:10] exclui o 10 -> linhas: 5


In [ ]:
# o padrao que voce mais vai usar
df_tipado.loc[df_tipado["uf"] == "SP", ["id_agendamento", "turno", "situacao"]].head()

   id_agendamento  turno   situacao
1        AG100001  Manha  Realizada
5        AG100005  Noite      Falta
8        AG100008  Tarde  Realizada
10       AG100010  Noite  Realizada
11       AG100011  Manha  Realizada

### A armadilha do encadeamento

**A regra: se há um colchete antes do outro, pare e reescreva com um único `.loc`.**


In [ ]:
teste = df_tipado.copy()

# ERRADO - o Pandas nao garante em qual objeto a alteracao caiu
teste[teste["uf"] == "SP"]["valor_consulta"] = 0
# -> dispara aviso de atribuicao encadeada (capturado abaixo) e NAO garante que a alteracao realmente aconteceu

ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment


> **Nota de versão:** nas versões mais recentes do Pandas (Copy-on-Write, como a 3.x usada neste
> notebook) esse aviso aparece como `ChainedAssignmentError` em vez do clássico `SettingWithCopyWarning`
> de versões anteriores — mas a causa e a correção são exatamente as mesmas: **um único `.loc`, nunca dois
> colchetes em sequência.**


In [ ]:
teste = df_tipado.copy()

# CERTO - um unico .loc, linha e coluna de uma vez
teste.loc[teste["uf"] == "SP", "valor_consulta"] = 0
print("agora funcionou de verdade:", (teste.loc[teste['uf']=='SP', 'valor_consulta'] == 0).all())

# trabalhando com um recorte? deixe explicito que e uma copia
sp = df_tipado.loc[df_tipado["uf"] == "SP"].copy()
sp["valor_consulta"] = 0        # agora nao ha ambiguidade
print("recorte explicito, sem warning, shape:", sp.shape)

agora funcionou de verdade: True
recorte explicito, sem warning, shape: (784, 14)


### Filtros

Filtro é uma máscara booleana: uma Series de `True`/`False` do mesmo tamanho do DataFrame. Use `na=False`
em `str.contains`/`str.match`: sem isso, linhas nulas viram `NaN` na máscara e o filtro quebra.


In [ ]:
# operadores logicos: & e |, sempre com parenteses em cada condicao
alto_valor_sp = df_tipado.loc[(df_tipado["uf"] == "SP") & (df_tipado["valor_consulta"] > 200)]
sp_ou_rj = df_tipado.loc[(df_tipado["uf"] == "SP") | (df_tipado["uf"] == "RJ")]
fora_sp_rj = df_tipado.loc[~df_tipado["uf"].isin(["SP", "RJ"])]

print("SP e valor > 200:", len(alto_valor_sp))
print("SP ou RJ        :", len(sp_ou_rj))
print("fora de SP/RJ   :", len(fora_sp_rj))

# pertencimento e intervalo
tres_ufs = df_tipado.loc[df_tipado["uf"].isin(["SP", "RJ", "MG"])]
adultos = df_tipado.loc[df_tipado["idade"].between(18, 59)]
print("SP/RJ/MG        :", len(tres_ufs))
print("idade 18-59     :", len(adultos))

# nulos
sem_renda = df_tipado.loc[df_tipado["renda_familiar"].isna()]
com_renda = df_tipado.loc[df_tipado["renda_familiar"].notna()]
print("renda ausente   :", len(sem_renda))
print("renda presente  :", len(com_renda))

# query: mais legivel em filtros longos
consulta_cara_manha = df_tipado.query("turno == 'Manha' and valor_consulta > 150 and idade >= 18")
print("query() manha, caro, adulto:", len(consulta_cara_manha))

SP e valor > 200: 259
SP ou RJ        : 1044
fora de SP/RJ   : 456
SP/RJ/MG        : 1309
idade 18-59     : 629
renda ausente   : 105
renda presente  : 1395
query() manha, caro, adulto: 333


### Filtro em dado sujo

**Um filtro em cima de dado não padronizado devolve um número errado sem gerar erro nenhum.**
`df["uf"] == "SP"` não pega `"sp"`, `" SP"` nem `"São Paulo"`. Antes de qualquer filtro em coluna de
texto, rode `.unique()` e olhe o que existe de verdade.

---

A coluna `uf` do nosso `df_tipado` foi convertida direto para `category` **sem canonicalização** — ou
seja, ela provavelmente já chegou "suja" de fábrica. Confirme isso com `.unique()`, mostre o efeito do
filtro ingênuo (`df["uf"] == "SP"`) vs. o filtro correto, e reporte a diferença de `shape`.


In [ ]:
print("valores distintos em 'uf' (categoria bruta):")
print(sorted(df_tipado["uf"].astype(str).unique()))

valores distintos em 'uf' (categoria bruta):
[' MG', ' RJ', ' SP', 'MG', 'RJ', 'S.P.', 'SP', 'São Paulo', 'mg', 'rj', 'sp']


In [ ]:
filtro_ingenuo = df_tipado.loc[df_tipado["uf"] == "SP"]

uf_canon = (df_tipado["uf"].astype(str)
                              .str.strip()
                              .str.upper()
                              .replace({"S.P.": "SP", "SÃO PAULO": "SP", "SAO PAULO": "SP"}))
filtro_correto = df_tipado.loc[uf_canon == "SP"]

print("filtro ingenuo  df['uf']=='SP' ->", filtro_ingenuo.shape[0], "linhas")
print("filtro correto  (canonicalizado) ->", filtro_correto.shape[0], "linhas")
print("linhas perdidas pelo filtro ingenuo:", filtro_correto.shape[0] - filtro_ingenuo.shape[0])

filtro ingenuo  df['uf']=='SP' -> 784 linhas
filtro correto  (canonicalizado) -> 908 linhas
linhas perdidas pelo filtro ingenuo: 124


**O número que "não faz sentido" é informação:** o filtro ingênuo subestimou a quantidade de
pacientes de SP porque tratou `"sp"`, `" SP"`, `"S.P."` e `"São Paulo"` como se fossem UFs diferentes de
`"SP"`. Depois do filtro, **sempre confira o `shape`** — foi assim que encontramos o problema.


---
# 4. `groupby`

O `groupby` não é um comando, é um modelo mental — **split, apply, combine**:

1. **Split** — divide por chave
2. **Apply** — calcula em cada grupo
3. **Combine** — junta os resultados


In [ ]:
# uma metrica, uma chave
print(df_tipado.groupby("uf", observed=True)["valor_consulta"].mean())

uf
 MG          188.064800
 RJ          168.430714
 SP          170.730303
MG           179.569094
RJ           179.166115
S.P.         172.173103
SP           181.095383
São Paulo    177.756190
mg           191.922308
rj           169.529333
sp           188.592439
Name: valor_consulta, dtype: float64


In [ ]:
# varias metricas de uma vez
df_tipado.groupby("uf", observed=True)["valor_consulta"].agg(["count", "mean", "median", "std"])

           count        mean   median        std
uf                                              
 MG           25  188.064800  185.170  37.737798
 RJ           14  168.430714  169.170  31.174898
 SP           33  170.730303  169.070  52.371969
MG           265  179.569094  179.870  47.796247
RJ           260  179.166115  176.120  43.500660
S.P.          29  172.173103  175.600  47.329574
SP           784  181.095383  180.405  45.530744
São Paulo     21  177.756190  177.760  37.067328
mg            13  191.922308  194.910  28.704186
rj            15  169.529333  180.060  40.979084
sp            41  188.592439  184.590  47.571883

In [ ]:
# duas chaves: resultado com indice hierarquico
df_tipado.groupby(["uf", "turno"], observed=True)["valor_consulta"].mean()

uf   turno 
 MG   Tarde    196.925000
     Manha     179.082222
     Noite     179.890000
     Tarde     202.102222
     manha     221.040000
                  ...    
sp   Noite     182.200000
     Tarde     192.385882
     manha     142.860000
     noite     231.490000
     tarde     208.760000
Name: valor_consulta, Length: 72, dtype: float64

In [ ]:
# nomeando as colunas de saida - o formato recomendado
resumo = df_tipado.groupby("turno", observed=True).agg(
    total_registros=("id_agendamento", "count"),
    valor_medio=("valor_consulta", "mean"),
    valor_total=("valor_consulta", "sum"),
    idade_mediana=("idade", "median"),
).reset_index()
resumo

    turno  total_registros  valor_medio  valor_total  idade_mediana
0   Tarde               43   184.410698      7929.66           43.0
1  MANHA                26   174.946538      4548.61           43.0
2   Manha              568   178.424454    101345.09           48.5
3   Manhã               20   190.568000      3811.36           47.0
4   NOITE               15   181.872667      2728.09           44.5
5   Noite              253   182.440870     46157.54           47.0
6   Tarde              510   179.729569     91662.08           46.5
7   manha               21   180.540000      3791.34           23.0
8   noite               11   202.767273      2230.44           23.0
9   tarde               33   185.861818      6133.44           47.0

### `agg`, `transform` e `filter`

| Método | O que devolve | Tamanho do resultado | Exemplo de uso |
|---|---|---|---|
| `agg` | Um valor por grupo | Uma linha por grupo | Média de valor por UF |
| `transform` | Um valor por linha original | Mesmo tamanho do DataFrame | Comparar cada registro com a média do seu grupo |
| `filter` | As linhas dos grupos aprovados | Subconjunto das linhas | Manter só as UFs com mais de 100 registros |
| `apply` | O que você escrever | Varia | Último recurso: é o mais lento |

### `transform` — o que resolve

A pergunta *"este registro está acima ou abaixo da média do grupo dele?"* não se responde com `agg`,
porque `agg` reduz. `transform` preserva.


In [ ]:
# media do grupo, repetida em cada linha do grupo
df_tipado["media_turno"] = df_tipado.groupby("turno", observed=True)["valor_consulta"].transform("mean")
df_tipado["desvio_da_media"] = df_tipado["valor_consulta"] - df_tipado["media_turno"]

df_tipado[["id_agendamento", "turno", "valor_consulta", "media_turno", "desvio_da_media"]].head()

  id_agendamento   turno  valor_consulta  media_turno  desvio_da_media
0       AG100000   Tarde          147.15   184.410698       -37.260698
1       AG100001   Manha          207.00   178.424454        28.575546
2       AG100002   Manha          264.04   178.424454        85.615546
3       AG100003   Manha          140.07   178.424454       -38.354454
4       AG100004   Tarde          275.33   179.729569        95.600431

In [ ]:
# manter apenas grupos (unidades) com volume relevante (>= 150 agendamentos)
df_rel = df_tipado.groupby("id_unidade", observed=True).filter(lambda g: len(g) >= 150)
print("linhas originais:", len(df_tipado), "| linhas apos filter() por volume de unidade:", len(df_rel))
print("unidades que sobraram:", sorted(df_rel['id_unidade'].unique()))

linhas originais: 1500 | linhas apos filter() por volume de unidade: 1500
unidades que sobraram: ['U01', 'U02', 'U03', 'U04', 'U05', 'U06', 'U07', 'U08']


### As armadilhas do `groupby`

- **Chaves nulas somem** — por padrão `dropna=True`: linhas cuja chave de agrupamento é nula desaparecem
  do resultado, e ninguém avisa.
- **Categorias vazias aparecem** — com colunas `category`, o `groupby` traz todas as categorias
  possíveis, inclusive as sem nenhum registro; use `observed=True` para trazer só as que existem.
- **A média das médias** ≠ a média geral, a menos que todos os grupos tenham o mesmo tamanho.
- **O índice vira chave** — sem `reset_index()`, o merge seguinte falha de um jeito confuso.
- **`count` e `size` não são a mesma coisa** — `count` conta valores não nulos da coluna; `size` conta
  linhas do grupo, nulas incluídas.

---
### Cinco armadilhas com números reais da nossa base:

1. Mostrando que agrupar por `idade` com `dropna=True` (padrão) esconde agendamentos com idade nula, e que
   `dropna=False` os revela.
2. Mostrando a diferença entre `groupby("uf")` com `observed=True` e `observed=False`.
3. Calculando a "média das médias" do `valor_consulta` por `turno` e compare com a média geral verdadeira.
4. Mostrando que `count` e `size` divergem em uma coluna com nulos.


In [ ]:
# 1) chaves nulas somem
com_dropna = df_tipado.groupby("idade", dropna=True)["id_agendamento"].count().sum()
sem_dropna = df_tipado.groupby("idade", dropna=False)["id_agendamento"].count().sum()
print("total contado com dropna=True (padrao) :", com_dropna)
print("total contado com dropna=False         :", sem_dropna)
print("agendamentos com idade nula que sumiram:", sem_dropna - com_dropna)

total contado com dropna=True (padrao) : 1456
total contado com dropna=False         : 1500
agendamentos com idade nula que sumiram: 44


In [ ]:
# 2) categorias vazias aparecem sem observed=True
grupos_observed_true = df_tipado.groupby("uf", observed=True)["valor_consulta"].mean()
grupos_observed_false = df_tipado.groupby("uf", observed=False)["valor_consulta"].mean()
print("observed=True  -> numero de grupos:", len(grupos_observed_true))
print("observed=False -> numero de grupos:", len(grupos_observed_false))
print()
print("com observed=False aparecem categorias sem nenhum registro (NaN na media):")
print(grupos_observed_false[grupos_observed_false.isna()])

observed=True  -> numero de grupos: 11
observed=False -> numero de grupos: 11

com observed=False aparecem categorias sem nenhum registro (NaN na media):
Series([], Name: valor_consulta, dtype: float64)


In [ ]:
# 3) media das medias vs media geral
media_das_medias = df_tipado.groupby("turno", observed=True)["valor_consulta"].mean().mean()
media_geral = df_tipado["valor_consulta"].mean()
print(f"media das medias por turno : {media_das_medias:.2f}")
print(f"media geral (correta)      : {media_geral:.2f}")
print(f"diferenca                  : {media_das_medias - media_geral:.2f}")

# forma correta de obter a media geral a partir de agregados: soma / contagem
agregado = df_tipado.groupby("turno", observed=True)["valor_consulta"].agg(["sum", "count"])
media_geral_via_agregado = agregado["sum"].sum() / agregado["count"].sum()
print(f"media geral via soma/contagem dos grupos: {media_geral_via_agregado:.2f}  (bate com a correta)")

media das medias por turno : 184.16
media geral (correta)      : 180.23
diferenca                  : 3.93
media geral via soma/contagem dos grupos: 180.23  (bate com a correta)


In [ ]:
# 4) count x size
contagem = df_tipado.groupby("turno", observed=True)["renda_familiar"].count()   # ignora nulos
tamanho = df_tipado.groupby("turno", observed=True)["renda_familiar"].size()     # conta tudo
comparacao = pd.DataFrame({"count_ignora_nulos": contagem, "size_conta_tudo": tamanho})
comparacao["nulos_de_renda"] = comparacao["size_conta_tudo"] - comparacao["count_ignora_nulos"]
comparacao

        count_ignora_nulos  size_conta_tudo  nulos_de_renda
turno                                                      
 Tarde                  40               43               3
MANHA                   23               26               3
Manha                  528              568              40
Manhã                   18               20               2
NOITE                   14               15               1
Noite                  232              253              21
Tarde                  477              510              33
manha                   21               21               0
noite                   11               11               0
tarde                   31               33               2

---
# 5. `merge`

### Os tipos de junção

| `how` | O que mantém | Quando usar | Risco |
|---|---|---|---|
| `inner` | Só o que existe nos dois lados | Quando a análise exige o par completo | Perde linhas em silêncio |
| `left` | Tudo da esquerda, o que casar da direita | Enriquecer uma base principal | Gera nulos onde não casou |
| `right` | O espelho do left | Raro — prefira inverter os lados e usar `left` | Confunde quem lê o código |
| `outer` | Tudo dos dois lados | Auditar diferenças entre duas fontes | Resultado maior que ambos |
| `cross` | Todas as combinações possíveis | Gerar grade de datas x unidades | Explode: N x M linhas |

### O parâmetro que salva a análise

`validate` declara qual cardinalidade você espera. Se a realidade for outra, o Pandas **interrompe** em
vez de multiplicar suas linhas em silêncio. `m:m` é o padrão quando você não declara nada — é também o
que produz tabelas com o triplo das linhas esperadas.


In [ ]:
# unidades.csv e esperado na mesma pasta do arquivo de agendamentos
caminho_unidades = caminho_agendamentos.parent / "unidades.csv"
unidades_ = pd.read_csv(caminho_unidades, sep=";", encoding="utf-8")
unidades_.head()

  id_unidade    nome_unidade     cidade uf_unidade cep_unidade
0        U01  Clínica Centro  São Paulo         SP   01310-100
1        U02   Clínica Norte  São Paulo         SP   02460-000
2        U03     Clínica Sul   Campinas         SP   13010-020
3        U04   Clínica Leste  Guarulhos         SP   07011-000
4        U05   Clínica Oeste  São Paulo         SP   05407-002

In [ ]:
# "muitos para um": a chave se repete na esquerda (varios agendamentos por unidade),
# e e unica na direita (uma linha por unidade) -- o caso mais comum ao enriquecer
enriquecido = pd.merge(df_tipado, unidades_, on="id_unidade", how="left", validate="m:1")
print("linhas antes do merge:", len(df_tipado), "| linhas depois:", len(enriquecido))
enriquecido[["id_agendamento", "id_unidade", "nome_unidade", "cidade"]].head()

linhas antes do merge: 1500 | linhas depois: 1500


  id_agendamento id_unidade       nome_unidade     cidade
0       AG100000        U01     Clínica Centro  São Paulo
1       AG100001        U07  Clínica Vila Nova     Osasco
2       AG100002        U06    Clínica Jardins  São Paulo
3       AG100003        U04      Clínica Leste  Guarulhos
4       AG100004        U04      Clínica Leste  Guarulhos

In [ ]:
# se a direita tiver duplicatas, validate="m:1" levanta erro -- vamos provocar isso de proposito
unidades_com_duplicata = pd.concat([unidades_, unidades_.iloc[[0]]], ignore_index=True)  # duplica a 1a unidade
print("linhas em unidades_com_duplicata:", len(unidades_com_duplicata), "(era", len(unidades_), ")")

pd.merge(df_tipado, unidades_com_duplicata, on="id_unidade", how="left", validate="m:1")

linhas em unidades_com_duplicata: 9 (era 8 )


[ERRO CAPTURADO PARA FINS DIDATICOS]
pandas.errors.MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 id_unidade
       U01 ...


O trecho acima **precisa** falhar — e falhou com `MergeError`, exatamente como esperado.
É isso que protege a análise: sem `validate`, esse merge teria simplesmente duplicado silenciosamente
todos os agendamentos da unidade U01.


In [ ]:
# indicator mostra de onde veio cada linha -- util para auditar duas fontes
r = pd.merge(df_tipado, unidades_, on="id_unidade", how="outer", indicator=True)
print(r["_merge"].value_counts())

_merge
both          1500
left_only        0
right_only       0
Name: count, dtype: int64


### Checklist do `merge`

**Antes:** as chaves têm o mesmo tipo nos dois lados? Estão canonicalizadas (acento, caixa, espaço)? A
chave é única onde você acha que é (`duplicated().sum()`)? Quantas linhas tem a tabela da esquerda?

**Depois:** o `shape` mudou como esperado? Quantos nulos apareceram nas colunas trazidas? Use
`indicator=True` e olhe o `value_counts`.

---
### Exercício 3

Rode o checklist completo do merge `df_tipado` (agendamentos) × `unidades_`:
1. Confirme que `id_unidade` tem o mesmo tipo nos dois lados.
2. Confirme que a chave é única em `unidades_` com `duplicated().sum()`.
3. Anote o número de linhas da esquerda antes do merge.
4. Depois do merge com `validate="m:1"`, confirme que o total de linhas não mudou.
5. Conte quantos nulos apareceram na coluna `nome_unidade` trazida pelo merge (mede o que não casou).

### Resolução


1) tipos das chaves:
   df_tipado['id_unidade']  dtype: str
   unidades_['id_unidade']  dtype: str

2) duplicatas de chave em unidades_: 0

3) linhas na esquerda antes do merge: 1500

4) linhas depois do merge: 1500 -> shape preservado? True

5) nulos em nome_unidade apos o merge: 0


### `merge`, `concat` e `join`

| Operação | O que faz | Comando |
|---|---|---|
| `merge` | Junta por valor de coluna — o SQL JOIN | `pd.merge(a, b, on="id", how="left")` |
| `concat` vertical | Empilha linhas de tabelas com as mesmas colunas | `pd.concat([jan, fev], ignore_index=True)` |
| `concat` horizontal | Cola colunas lado a lado, alinhando pelo índice | `pd.concat([a, b], axis=1)` |
| `join` | Junta pelo índice, não por coluna | `a.join(b, how="left")` |

`concat` horizontal alinha pelo índice: **se os índices não corresponderem, o resultado é uma tabela
cheia de nulos que parece certa.**


In [ ]:
jan = df_tipado[df_tipado["dt_agendamento"].dt.month == 1]
fev = df_tipado[df_tipado["dt_agendamento"].dt.month == 2]
empilhado = pd.concat([jan, fev], ignore_index=True)
print("jan:", len(jan), "| fev:", len(fev), "| empilhado:", len(empilhado))

jan: 248 | fev: 249 | empilhado: 497


In [ ]:
# armadilha do concat horizontal: indices que nao correspondem geram nulos silenciosos
a = df_tipado[["id_agendamento", "turno"]].head(3)
b = df_tipado[["valor_consulta"]].sample(3, random_state=1)   # embaralha o indice de proposito

print("indice de a:", list(a.index))
print("indice de b:", list(b.index))
pd.concat([a, b], axis=1)

indice de a: [0, 1, 2]
indice de b: [91, 75, 1264]


     id_agendamento   turno  valor_consulta
0          AG100000   Tarde             NaN
1          AG100001   Manha             NaN
2          AG100002   Manha             NaN
91              NaN     NaN           95.59
75              NaN     NaN          142.73
1264            NaN     NaN          149.15

Note como o `concat` horizontal juntou `a` e `b` **pelo índice**, e não pela posição — como os
índices de `b` não batem com os de `a`, a coluna `valor_consulta` aparece cheia de `NaN` nas linhas de
`a` e as linhas de `b` viram novas linhas. É exatamente a armadilha descrita na teoria: **o resultado parece uma
tabela normal, mas está errado.** A correção é resetar os índices antes (`.reset_index(drop=True)`) ou,
melhor ainda, usar `merge`/`join` por uma chave explícita.


---
# 6. Primeiro diagnóstico de nulos

Já vimos anteriormente como **tratar** valores ausentes. Agora o objetivo é anterior e mais importante:
**entender o que está faltando, quanto, onde e por quê** — antes de decidir qualquer coisa.

> **Tratar nulo antes de diagnosticar é apagar a evidência de um problema que ainda não foi compreendido.**

Todo diagnóstico realizado aqui deve virar linha no dicionário de dados (Seção 7).

### O relatório de nulos

Uma função, três perguntas: quanto falta em cada coluna, quanto falta em cada linha, e o que falta junto.


In [ ]:
def diagnostico_nulos(df):
    rel = pd.DataFrame({
        "nulos": df.isna().sum(),
        "pct": (df.isna().mean() * 100).round(1),
        "distintos": df.nunique(dropna=True),
        "tipo": df.dtypes.astype(str),
    })
    return rel.sort_values("pct", ascending=False)

diagnostico_nulos(df_tipado)

                 nulos  pct  distintos            tipo
renda_familiar     105  7.0       1363         float64
cep_paciente        60  4.0       1440             str
data_nascimento     58  3.9       1392  datetime64[us]
idade               44  2.9         95           Int64
id_agendamento       0  0.0       1500             str
sexo                 0  0.0          3        category
uf                   0  0.0         11        category
id_paciente          0  0.0        716             str
id_unidade           0  0.0          8             str
turno                0  0.0         10        category
dt_consulta          0  0.0       1371  datetime64[us]
dt_agendamento       0  0.0       1378  datetime64[us]
situacao             0  0.0          3        category
valor_consulta       0  0.0       1422         float64
media_turno          0  0.0         10         float64
desvio_da_media      0  0.0       1472         float64

In [ ]:
# quantas colunas faltam em cada linha
df_tipado["qtd_nulos_linha"] = df_tipado.isna().sum(axis=1)
df_tipado["qtd_nulos_linha"].value_counts().sort_index()

qtd_nulos_linha
0    1245
1     243
2      12
Name: count, dtype: int64

In [ ]:
# colunas que costumam faltar juntas
colunas_numericas_com_nulo = ["data_nascimento", "idade", "renda_familiar", "valor_consulta"]
df_tipado[colunas_numericas_com_nulo].isna().corr(numeric_only=False).round(2)

                 data_nascimento  idade  renda_familiar  valor_consulta
data_nascimento             1.00  -0.01           -0.04             NaN
idade                      -0.01   1.00           -0.02             NaN
renda_familiar             -0.04  -0.02            1.00             NaN
valor_consulta               NaN    NaN             NaN             NaN

### Nulos que não parecem nulos

O `isna()` só enxerga o que o Pandas reconheceu como ausente **na leitura**. A maior parte da ausência
real está disfarçada de valor:

| Disfarce | Onde aparece | Como encontrar |
|---|---|---|
| String vazia `""` | Exportações de banco e de Excel | `df[col].str.strip().eq("").sum()` |
| Texto de ausência | `"N/A"`, `"-"`, `"não informado"`, `"SEM DADO"` | `value_counts()` nas colunas de texto |
| Zero como ausência | Renda 0, idade 0, peso 0 | Comparar a contagem de zeros com o plausível |
| Sentinela numérica | 999, 9999, -1, 99999999 | `describe()`: o máximo denuncia |
| Data impossível | 01/01/1900, 31/12/9999 | `min()` e `max()` da coluna de data |
| Espaço em branco | `" "` e o `\xa0` do copiar e colar | `str.strip()` antes de qualquer contagem |
| Código de sistema | `"XX"`, `"ZZZZ"`, `"000"` | Cruzar com a tabela de domínio do dicionário |

---
### Exercício 4

Releia o CSV **cru** (`df`, ainda todo `object`/string, sem os `na_values` amplos) e rode a varredura de
disfarçados vista na teoria para encontrar sentinelas escondidas em `renda_familiar` e `idade`. Em seguida
confirme, usando `describe()`, a sentinela numérica que aparece em `renda_familiar` depois de convertida.

### Resolução


In [ ]:
# releitura minima, sem tratar quase nenhum na_values, para simular a "primeira olhada" antes do diagnostico
df_cru = pd.read_csv(caminho_agendamentos, sep=";", encoding="latin1", dtype=str)


data_nascimento: 26 valores suspeitos
idade: 30 valores suspeitos
cep_paciente: 39 valores suspeitos
renda_familiar: 24 valores suspeitos


See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.


In [ ]:
# sentinelas numericas: o maximo costuma entregar


                    min         max
renda_familiar_num -1.0  99999999.0


### O padrão da ausência é informação

Nulo espalhado ao acaso é um problema técnico. **Nulo concentrado é um achado.** Perguntas que valem
fazer antes de decidir o tratamento: a ausência se concentra em algum período? Em alguma unidade? Duas
colunas faltam sempre juntas? A ausência se relaciona com o desfecho que você quer analisar?


In [ ]:
# a ausencia de 'idade' se concentra em algum periodo?
nulos_por_mes = (df_tipado.assign(mes=df_tipado["dt_agendamento"].dt.to_period("M"))
                            .groupby("mes", observed=True)["idade"]
                            .apply(lambda s: s.isna().mean() * 100).round(1))
nulos_por_mes

mes
2024-01    17.7
2024-02     0.0
2024-03     0.0
2024-04     0.0
2024-05     0.0
2024-06     0.0
Freq: M, Name: idade, dtype: float64

Confirma-se exatamente o padrão descrito no exemplo de campo documentado: **a ausência de
`idade` se concentra em janeiro/2024**, consistente com uma hipótese de troca de sistema. Essa observação
— "2,3% nulos, concentrados em jan/2024 — troca de sistema" — é *exatamente* o tipo de frase que vai para
a linha **Qualidade observada** do dicionário de dados na próxima seção.


---
# 7. Dicionário de dados

Em uma etapa anterior do projeto, o grupo já deve ter documentado alguns campos, com papéis definidos —
um exercício de profundidade. Agora o
dicionário passa a ser um **artefato vivo do projeto**, e ganha uma dimensão que ficou de fora:

> **granularidade · tipo · domínio · regra · origem**

### Granularidade

A pergunta mais barata de fazer e a mais cara de esquecer: **o que representa uma linha desta tabela?**
A granularidade (ou grão) é a unidade de observação. Precisa estar escrita no topo do dicionário, antes
de qualquer campo.

| Tabela | Grão | Chave que identifica a linha |
|---|---|---|
| Pedidos | Um pedido | id_pedido |
| Itens do pedido | Um produto dentro de um pedido | id_pedido + id_produto |
| Agendamentos | Uma consulta marcada | id_agendamento |
| Histórico de status | Uma mudança de status de uma consulta | id_agendamento + data_hora_status |
| Faturamento mensal | Um cliente em um mês | id_cliente + ano_mes |

### Grão errado, soma errada

O erro clássico, e ele não gera nenhuma mensagem: uma tabela que repete um valor "de cabeçalho" em cada
linha, e alguém soma essa coluna. **O resultado é um número plausível, alto e completamente errado.**
Como se protege: escreva o grão no dicionário e confira se a chave declarada é realmente única
(`duplicated(subset=chave).sum()` deve dar zero).

---

Confirmando o grão declarado da nossa tabela (`id_agendamento` é único) e, em seguida, **simulando o erro
clássico de grão errado**, criando uma tabela de "itens" fictícia onde o valor total do agendamento aparece
repetido a cada item, e mostrando como somar essa coluna sem cuidado infla o faturamento.

### Resolução


In [ ]:
print("id_agendamento e unico?", df_tipado['id_agendamento'].duplicated().sum() == 0,
      "-> duplicatas:", df_tipado['id_agendamento'].duplicated().sum())

id_agendamento e unico? True -> duplicatas: 0


In [ ]:
# simulando uma tabela "de itens" no grao errado: cada agendamento vira 1 a 3 "itens de procedimento",
# repetindo o valor total do agendamento em cada item (erro classico de modelagem)
rng_demo = np.random.default_rng(123)
base_exemplo = df_tipado.head(200).copy()
n_itens = rng_demo.integers(1, 4, size=len(base_exemplo))
itens = base_exemplo.loc[base_exemplo.index.repeat(n_itens)].reset_index(drop=True)
itens["id_item"] = itens.groupby("id_agendamento").cumcount() + 1

faturamento_real = base_exemplo["valor_consulta"].sum()
faturamento_errado = itens["valor_consulta"].sum()   # soma no grao errado (grao = item, nao agendamento)

print(f"faturamento real (grao = agendamento) : R$ {faturamento_real:,.2f}")
print(f"faturamento com grao errado (=item)   : R$ {faturamento_errado:,.2f}")
print(f"inflado em                            : {faturamento_errado / faturamento_real - 1:.1%}")
print()
print("checagem de unicidade que teria pegado o problema:")
print("  duplicated(subset='id_agendamento').sum() em 'itens':", itens.duplicated(subset="id_agendamento").sum())

faturamento real (grao = agendamento) : R$ 35,990.89
faturamento com grao errado (=item)   : R$ 69,965.74
inflado em                            : 94.4%

checagem de unicidade que teria pegado o problema:
  duplicated(subset='id_agendamento').sum() em 'itens': 186


**Exatamente o alerta da teoria:** a chave `id_agendamento` **não** é única na tabela `itens` (o
`duplicated().sum()` deu maior que zero) — isso é o sinal de alarme que deveria ter impedido a soma direta
de `valor_consulta`. O jeito certo seria agregar por `id_agendamento` primeiro (`.drop_duplicates()` ou
`groupby("id_agendamento")["valor_consulta"].first()`) antes de somar.

### As cinco dimensões de cada campo

| Dimensão | O que documenta |
|---|---|
| **Granularidade** | Em que grão o campo faz sentido, e se ele é aditivo |
| **Tipo** | Físico (no arquivo), lógico (após conversão), semântico (identificador, e não número) |
| **Domínio** | O conjunto de valores válidos: lista, faixa, padrão ou referência |
| **Regra** | O que precisa ser verdade: obrigatoriedade, unicidade, relação, restrição de negócio |
| **Origem** | Sistema, tabela e campo de onde veio, e que transformação sofreu no caminho |

### Os quatro tipos de domínio

| Tipo de domínio | Como se declara | Exemplo |
|---|---|---|
| Enumerado | A lista completa dos valores válidos | `sexo`: M, F, O |
| Faixa | Limite inferior e superior | `idade`: 0 a 120 |
| Padrão | Expressão regular ou máscara | `cep`: `^\d{8}$` |
| Referência | Aponta para a chave de outra tabela | `id_unidade`: tabela `unidades` |

Domínio declarado é o que permite escrever a regra de validação. Sem domínio, "validar" vira opinião.

---

Documentando **de verdade** cinco campos da nossa base `agendamentos`, no mesmo formato do exemplo `dt_agend` visto na teoria
(`dt_agend`), usando o que descobrimos nas seções anteriores (tipo de disfarce de nulo, concentração de
janeiro/2024, sentinela de renda, canonicalização de `uf` etc.). Montamos isso como uma tabela (DataFrame),
para que vire de fato um artefato reutilizável do projeto.

### Resolução


In [ ]:
dicionario = pd.DataFrame([
    {
        "nome_fisico": "dt_agendamento",
        "nome_negocio": "Data e hora do agendamento",
        "descricao": "Momento em que a consulta foi marcada no sistema, não o momento da consulta em si",
        "granularidade": "Um agendamento; não aditivo",
        "tipo_fisico_logico_semantico": "texto no CSV / datetime64 / data de evento",
        "dominio": "Entre 01/01/2024 e a data atual; horário comercial das unidades",
        "regra": "Obrigatório; deve ser anterior a dt_consulta",
        "origem": "Sistema fictício SGA-Clínicas, tabela AGENDA, campo DT_AGEND; convertido de DD/MM/AAAA HH:MM",
        "qualidade_observada": "0% nulos (obrigatório na origem)",
        "sensibilidade": "Interno",
    },
    {
        "nome_fisico": "idade",
        "nome_negocio": "Idade do paciente na data de referência",
        "descricao": "Calculada a partir de data_nascimento; usada para faixa etária nas análises de falta",
        "granularidade": "Um agendamento; não aditivo (não se soma idade)",
        "tipo_fisico_logico_semantico": "texto no CSV / Int64 / atributo descritivo",
        "dominio": "Faixa: 0 a 120",
        "regra": "Obrigatório; deve ser consistente com data_nascimento",
        "origem": "Derivado de data_nascimento, sistema SGA-Clínicas",
        "qualidade_observada": "cerca de 18% nulos em jan/2024 (concentrados) — hipótese: troca de sistema",
        "sensibilidade": "Pessoal",
    },
    {
        "nome_fisico": "renda_familiar",
        "nome_negocio": "Renda familiar declarada",
        "descricao": "Autodeclarada no cadastro; usada em análises socioeconômicas de acesso",
        "granularidade": "Um agendamento (repete por paciente); não aditivo",
        "tipo_fisico_logico_semantico": "texto no CSV / float64 / atributo descritivo",
        "dominio": "Faixa: R$ 300 a R$ 30.000 (valores fora disso são sentinela, não dado real)",
        "regra": "Opcional; 99999999 e -1 são sentinelas de 'não informado' e devem virar nulo",
        "origem": "Sistema SGA-Clínicas, cadastro do paciente",
        "qualidade_observada": "~7% nulos diretos + sentinela numérica 99999999 disfarçando ausência",
        "sensibilidade": "Restrito",
    },
    {
        "nome_fisico": "uf",
        "nome_negocio": "UF de residência do paciente",
        "descricao": "Unidade federativa declarada no cadastro do paciente",
        "granularidade": "Um agendamento; não aditivo",
        "tipo_fisico_logico_semantico": "texto no CSV / category / atributo categórico",
        "dominio": "Enumerado: SP, RJ, MG (siglas oficiais, 2 letras maiúsculas)",
        "regra": "Obrigatório; precisa ser canonicalizado antes de qualquer filtro/groupby",
        "origem": "Sistema SGA-Clínicas, cadastro do paciente",
        "qualidade_observada": "~12% dos valores em variantes não canônicas (minúsculo, espaço, nome extenso)",
        "sensibilidade": "Interno",
    },
    {
        "nome_fisico": "situacao",
        "nome_negocio": "Situação final do agendamento",
        "descricao": "Resultado do agendamento: se o paciente compareceu, faltou ou cancelou antes",
        "granularidade": "Um agendamento; não aditivo",
        "tipo_fisico_logico_semantico": "texto no CSV / category / atributo categórico",
        "dominio": "Enumerado: Realizada, Falta, Cancelada",
        "regra": "Obrigatório; é a base do cálculo de taxa de falta (métrica da pergunta de PI)",
        "origem": "Sistema SGA-Clínicas, tabela AGENDA, campo STATUS",
        "qualidade_observada": "0% nulos",
        "sensibilidade": "Interno",
    },
])
dicionario

      nome_fisico                             nome_negocio                                          descricao  \
0  dt_agendamento               Data e hora do agendamento  Momento em que a consulta foi marcada no siste...   
1           idade  Idade do paciente na data de referência  Calculada a partir de data_nascimento; usada p...   
2  renda_familiar                 Renda familiar declarada  Autodeclarada no cadastro; usada em análises s...   
3              uf             UF de residência do paciente  Unidade federativa declarada no cadastro do pa...   
4        situacao            Situação final do agendamento  Resultado do agendamento: se o paciente compar...   

                                       granularidade                   tipo_fisico_logico_semantico  \
0                        Um agendamento; não aditivo     texto no CSV / datetime64 / data de evento   
1    Um agendamento; não aditivo (não se soma idade)     texto no CSV / Int64 / atributo descritivo   
2  Um agenda

In [ ]:
caminho_dicionario = caminho_agendamentos.parent / "dicionario_de_dados.csv"
dicionario.to_csv(caminho_dicionario, index=False, encoding="utf-8")
print("Dicionário salvo em:", caminho_dicionario.resolve())

Dicionário salvo em: /home/claude/nb2/dicionario_de_dados.csv


---
# 8. Catálogo, linhagem e governança

### Dicionário e catálogo não são a mesma coisa

| | Dicionário de dados | Catálogo de dados |
|---|---|---|
| Pergunta que responde | O que significa este campo? | Que dados existem, e onde? |
| Escopo | Uma base, campo a campo | Todas as bases da organização |
| Quem lê | Quem vai usar aquela base | Quem ainda não sabe qual base usar |
| Profundidade | Alta, sobre poucos objetos | Menor, sobre muitos objetos |
| Analogia | A bula do remédio | A farmácia inteira, com o índice |

O catálogo **contém** os dicionários. Um dicionário sem catálogo é um documento que ninguém encontra.

### O que um catálogo guarda

- **Metadados técnicos** — nome do objeto, colunas, tipos, tamanho, particionamento, frequência de atualização, última carga
- **Metadados de negócio** — o que a base significa, quem é o dono, que domínio ela cobre, que perguntas ela responde
- **Metadados operacionais** — quem consome, com que frequência, SLA, histórico de incidentes, qualidade medida
- **Metadados de governança** — classificação de sensibilidade, base legal de tratamento, política de retenção, quem pode conceder acesso

### Quem procura o quê em um catálogo

| Quem | O que procura | Sem catálogo, o que acontece |
|---|---|---|
| Analista novo | Qual base tem o dado de que preciso | Pergunta no corredor; usa a que alguém indicou |
| Cientista de dados | O dado é confiável? Com que frequência atualiza? | Descobre o problema depois de treinar o modelo |
| Engenheiro de dados | Se eu mudar esta coluna, o que quebra? | Descobre quando quebra |
| Jurídico e compliance | Onde há dado pessoal, e com que base legal | Não consegue responder a um pedido de titular |
| Gestor | Por que dois relatórios dão números diferentes | Reunião de três horas sem conclusão |

---
### Exercício 4 — construindo um mini-catálogo

Monte um registro de catálogo (nível "objeto", não "campo") para a nossa tabela `agendamentos`, cobrindo
as quatro camadas de metadados acima.

### Resolução


In [ ]:
catalogo_agendamentos = {
    # metadados tecnicos
    
    # metadados de negocio
    
    # metadados operacionais
   
    # metadados de governanca
  
}



nome_objeto                 : agendamentos.csv
colunas                     : ['id_agendamento', 'id_paciente', 'data_nascimento', 'idade', 'sexo', 'uf', 'cep_paciente', 'id_unidade', 'turno', 'dt_agendamento', 'dt_consulta', 'situacao', 'valor_consulta', 'renda_familiar']
linhas_estimadas            : 1500
particionamento             : nenhum (arquivo único, escopo = 1º semestre de 2024)
frequencia_atualizacao      : diária (simulada)
ultima_carga                : 2024-06-30
significado                 : Um agendamento de consulta em uma unidade da rede de clínicas
data_owner                  : Gestor de Operações Assistenciais
dominio_de_negocio          : Atendimento / Agendamento
perguntas_que_responde      : ['Taxa de falta por turno e faixa etária', 'Volume de agendamentos por unidade']
consumidores                : ['Time de BI', 'Grupo de PI da disciplina de dados']
sla_atualizacao             : D+1
qualidade_medida            : 18% de nulos em idade concentrados em jan/2024 (tr

### Linhagem: o caminho do dado

Linhagem é o registro de por onde o dado passou e o que aconteceu com ele em cada etapa: **origem** →
**ingestão** → **transformação** → **consumo**. Não é só o desenho do pipeline: é a capacidade de
identificar de onde veio um número específico apresentado em um relatório.

Para que serve: **análise de impacto** (o que quebra se eu mudar esta coluna), **depuração** (em qual
etapa o número saiu errado), **auditoria e conformidade**, e **confiança** — *um número cuja origem pode
ser demonstrada é discutível; um número sem origem é apenas uma opinião com formatação.*

---
### Exercício 5 — registrando a linhagem deste notebook

Documente, em uma pequena tabela, a linhagem real do indicador **"taxa de falta por turno"** que
calculamos na Seção 4, do sistema de origem até o número final.

### Resolução


             etapa                                          descricao
0           Origem   Sistema transacional SGA-Clínicas, tabela AGENDA
1         Ingestão  Exportação diária para agendamentos.csv (sep='...
2  Transformação 1  Leitura em duas etapas (amostra + leitura comp...
3  Transformação 2  Conversão de tipos com intenção: situacao/turn...
4  Transformação 3        groupby('turno').agg(...) com observed=True
5          Consumo  Tabela 'resumo' exibida na Seção 4 deste noteb...

### Governança — quem responde pelo quê

| Papel | Responsabilidade | Quem costuma ser |
|---|---|---|
| **Data Owner** | Responde pelo dado perante o negócio: aprova acesso, define regras e domínios | Gestor da área que gera o dado |
| **Data Steward** | Cuida da qualidade no dia a dia: mantém o dicionário, apura inconsistências | Analista da área |
| **Data Custodian** | Cuida da infraestrutura: armazenamento, backup, segurança técnica | Engenharia de dados ou TI |
| **Data Consumer** | Usa o dado; reporta problemas encontrados | Analistas, cientistas, áreas de negócio |

O papel mais esquecido é o de **Owner**. Sem alguém que responda pelo dado, toda inconsistência vira
problema de ninguém.

### Classificação e LGPD

| Classificação | O que é |
|---|---|
| **Público** | Pode ser divulgado livremente |
| **Interno** | Circula na organização, não sai dela |
| **Restrito** | Acesso por necessidade demonstrada (estratégico, financeiro, contratual) |
| **Pessoal** | Identifica uma pessoa natural; exige base legal, finalidade declarada e prazo de retenção |
| **Pessoal sensível** | Saúde, biometria, origem racial, religião, opinião política, orientação sexual, filiação sindical |

Se o projeto toca dado pessoal, isso precisa estar declarado no dicionário, **campo a campo**.

---
### Exercício 6 — classificando cada coluna da base

Classifique cada coluna de `agendamentos` segundo a tabela de sensibilidade acima. (Note que, por ser uma
base de **saúde** — o fato de a pessoa ter um agendamento clínico e sua situação de comparecimento —
algumas colunas se aproximam de dado pessoal sensível; discuta isso no seu projeto real com cuidado.)

### Resolução


                         coluna     classificacao                                             motivo
0                id_agendamento           Interno  Identificador técnico, não identifica pessoa s...
1                   id_paciente           Pessoal  Chave que permite rastrear todos os registros ...
2               data_nascimento           Pessoal                                Dado pessoal direto
3                          sexo           Pessoal                                   Atributo pessoal
4             uf / cep_paciente           Pessoal     Localização que pode identificar indiretamente
5            id_unidade / turno           Interno          Dado operacional, não identifica paciente
6  dt_agendamento / dt_consulta  Pessoal sensível       Associa uma pessoa a um atendimento de saúde
7                      situacao  Pessoal sensível  Revela comportamento de saúde (compareceu/falt...
8                valor_consulta          Restrito             Dado financeiro/operacional d

### Ferramentas de catálogo

- **Open source:** OpenMetadata, DataHub, Amundsen, Apache Atlas — coletam metadados automaticamente das fontes e montam linhagem
- **Comerciais e de nuvem:** Collibra, Alation, Microsoft Purview, AWS Glue Data Catalog, Google Dataplex
- **Para um projeto acadêmico:** uma planilha bem estruturada, versionada no repositório junto com o código, resolve. O que importa não é a ferramenta: é **o hábito de registrar enquanto se trabalha**.

### Onde a documentação entra no trabalho

A ordem que a maioria segue: *analisar, entregar, e documentar se sobrar tempo* — que nunca sobra. A
ordem que funciona coloca a documentação **no meio**, não no fim:

**Ler a base → Diagnosticar tipos e nulos → Documentar no mesmo dia → Tratar com critério → Analisar e responder**

O que você descobre ao diagnosticar já é o conteúdo do dicionário — foi exatamente o que fizemos neste
notebook: cada achado das Seções 2, 3 e 6 virou uma linha do dicionário na Seção 7.

---
## O que levar para o PI

Ao final desta etapa, o grupo deve apresentar quatro coisas:

1. **A base carregada**, com os tipos convertidos conscientemente e o que não converteu registrado (Seção 2)
2. **A matriz pergunta → métrica → colunas → operação**, com a coluna "tem na base?" preenchida com honestidade (Seção 1)
3. **O relatório de nulos**, incluindo os disfarçados, com hipótese sobre o padrão de cada ausência relevante (Seção 6)
4. **O dicionário ampliado**: granularidade declarada no topo, e as cinco dimensões em cada campo documentado (Seção 7)

Se a base não responder à pergunta do projeto, isso também é um resultado — desde que venha acompanhado
de qual das três saídas legítimas o grupo escolheu: **enriquecer, trocar a base, ou ajustar a pergunta**.
